# 07c · Pilot diagnostics on the GPU

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 1 (pilot follow-up)

**Purpose.** Three checks that need new engines: how much of the network TensorRT actually runs in INT8, how much results move between identical rebuilds, and whether an extreme calibration set changes results at all.

**In plain words.** The pilot only set the INT8 flag, which allows INT8 but lets TensorRT keep any layer in FP32 when that is faster; calibration ranges of FP32 layers are never used. We rebuild the natural seed-1 engine with per-layer profiling (three times, which also measures rebuild noise) and build two deliberately extreme calibration sets as positive controls: bona fide only, and a tiny set of 20 images.

**Expected inputs**
- Notebook 07 outputs (pilot engine `int8_entropy_natural_s1`, calibration manifest); ONNX model and `equivalence.json`; processed train/dev/test frames; FP32 `threshold.json`. Same GPU type and TensorRT version as the pilot.

**Expected outputs**
- `QPAD_Project/engines/diagnostics/` and `QPAD_Project/results/int8/diagnostics/` (`layer_precision_*.csv`, `precision_summary.csv`, `rebuild_noise.json`, `control_*.csv`, `control_manifest.csv`)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [13]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

qpad 0.1.0 from /content/drive/MyDrive/QPAD_Project/code/qpad


In [15]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

DATASET_ROOT       = /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw
  train    -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/Train_files-002.tar
  dev      -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/Dev_files-001.tar
  test     -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/Test_files-003.tar
  metadata -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/OULU-NPU-20261010T032906Z-1-004.zip
PROJECT_DRIVE_ROOT = /content/drive/MyDrive/QPAD_Project
LOCAL_WORK_ROOT    = /content/qpad_work


In [16]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

{
 "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
 "packages": {
  "python": "3.13.16",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "cv2": "4.14.0",
  "onnx": "1.23.2",
  "onnxruntime": "1.31.0",
  "tensorrt": "10.16.1.11",
  "modelopt": null,
  "scipy": "1.16.3",
  "matplotlib": "3.10.0",
  "yaml": "6.0.3"
 },
 "gpu": {
  "cuda_available": true,
  "torch_cuda": "13.0",
  "cudnn": 92700,
  "gpu_name": "NVIDIA A100-SXM4-40GB",
  "gpu_count": 1,
  "nvidia_smi": [
   "NVIDIA A100-SXM4-40GB, 580.82.07, 40960 MiB"
  ],
  "driver_version": "580.82.07"
 }
}
git: {'commit': None, 'dirty': None}
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   60G   54G  53% /



In [17]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = False                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
DRY_RUN_EVAL_VIDEOS_PER_CLASS = 3   # dry run: one diagnostic engine, evaluated on 3 videos/class/split
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb07c", paths.logs / "nb07c.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

2026-10-31 04:37:26,893 INFO qpad.nb07c: DRY_RUN=False seed=42
Outputs of this run go to: /content/drive/MyDrive/QPAD_Project


In [18]:
# TensorRT is needed from here on. Its version is pinned in configs/int8_experiments.yaml: TensorRT 11
# removed the INT8 entropy calibrator that the pre-registered method uses. Colab preinstalls TensorRT 11,
# so the first time this cell swaps it for the pinned version and asks for ONE session restart.
TRT_PIN = str(cfgs["int8"]["tensorrt"]["version"])
def installed_trt():   # checked in a fresh process: an already-imported module would hide a reinstall
    r = subprocess.run([sys.executable, "-c", "import tensorrt; print(tensorrt.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None
if IN_COLAB and installed_trt() != TRT_PIN:
    import torch
    cuda_major = torch.version.cuda.split(".")[0]
    old = [p["name"] for p in json.loads(subprocess.run([sys.executable, "-m", "pip", "list", "--format=json"],
                                                        capture_output=True, text=True, check=True).stdout)
           if p["name"].lower().replace("_", "-").startswith("tensorrt")]
    if old:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *old], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"tensorrt-cu{cuda_major}=={TRT_PIN}"], check=True)
    if installed_trt() != TRT_PIN:
        raise RuntimeError(f"installing TensorRT {TRT_PIN} failed (found {installed_trt()})")
    if "tensorrt" in sys.modules:
        raise RuntimeError(f"TensorRT {TRT_PIN} is now installed. Runtime -> Restart session, then Run all again "
                           "(needed once per runtime; the old version is still loaded in this session).")
try:
    import tensorrt as trt
except ImportError:
    trt = None
print("TensorRT:", getattr(trt, "__version__", None))
if trt is not None and trt.__version__ != TRT_PIN:
    raise RuntimeError(f"TensorRT {trt.__version__} is loaded but {TRT_PIN} is pinned: restart the session.")
from qpad.tensorrt_utils import hardware_fingerprint
HW = hardware_fingerprint(); print(HW)

TensorRT: 10.16.1.11
{'gpu_name': 'NVIDIA A100-SXM4-40GB', 'driver_version': '580.82.07', 'torch_cuda': '13.0', 'cudnn': 92700, 'tensorrt': '10.16.1.11', 'torch': '2.11.0+cu130'}


In [19]:
import json, torch, pandas as pd
from qpad import diagnostics as D
from qpad.calibration import attach_calibration_frames, compare_scales, nested_calibration_sets, scale_summary
from qpad.data_exceptions import expected_eval_counts
from qpad.datasets import materialize, video_index
from qpad.experiments import (balanced_subset, build_calibration_manifest, build_int8_engine, calibration_array,
                              engine_is_complete, engine_tag, evaluate_backend, rebuild_noise)
from qpad.metrics import metrics_rows
from qpad.pipeline import find_input, processed_split_dir
from qpad.tensorrt_utils import TrtRunner, engine_layers
if not torch.cuda.is_available():
    raise RuntimeError("These diagnostics need a CUDA GPU runtime.")
icfg, dcfg = cfgs["int8"], cfgs["int8"]["diagnostics"]
METHOD = icfg["calibration_method"]["primary"]
REF_TAG = engine_tag(METHOD, "natural", 1)
REF_ENGINE = find_input(paths, DRY_RUN, "engines", REF_TAG)
REF_RESULT = find_input(paths, DRY_RUN, "results", f"int8/{REF_TAG}")
pilot_hw = json.loads((REF_ENGINE / "engine_meta.json").read_text())["hardware"]
if (pilot_hw["gpu_name"], pilot_hw["tensorrt"]) != (HW["gpu_name"], HW["tensorrt"]):
    raise RuntimeError(f"Pilot engines were built on {pilot_hw['gpu_name']} with TensorRT {pilot_hw['tensorrt']}, "
                       f"this runtime has {HW['gpu_name']} with TensorRT {HW['tensorrt']}. Use the same GPU type.")
DIAG = OUT.results / "int8" / "diagnostics"
DIAG.mkdir(parents=True, exist_ok=True)
print("reference pilot engine:", REF_TAG, "| writing to:", DIAG)

reference pilot engine: int8_entropy_natural_s1 | writing to: /content/drive/MyDrive/QPAD_Project/results/int8/diagnostics


In [20]:
norm = cfgs["preprocessing"]["normalization"]
nfr = expected_eval_counts(cfgs)
CAL, CAL_STORE, CAL_PATH = build_calibration_manifest(paths, cfgs, DRY_RUN, [1], ["natural"])
n_eval = DRY_RUN_EVAL_VIDEOS_PER_CLASS if DRY_RUN else None
DEV = balanced_subset(materialize(processed_split_dir(paths, DRY_RUN, "dev"), "dev", "eval", OUT.local_cache), n_eval)
TEST = balanced_subset(materialize(processed_split_dir(paths, DRY_RUN, "test"), "test", "eval", OUT.local_cache), n_eval)
FP32_THR = json.loads(find_input(paths, DRY_RUN, "results", "fp32/threshold.json").read_text())["threshold"]
CONTROLS = {}
if not DRY_RUN:
    train_videos = video_index(CAL_STORE)
    for name, mix in dcfg["controls"].items():
        selection = nested_calibration_sets(train_videos, {name: mix}, [1], sum(mix.values()))
        CONTROLS[name] = attach_calibration_frames(selection, CAL_STORE.index,
                                                   int(icfg["calibration_budget"]["frames_per_video"]))
    CONTROL_MANIFEST = pd.concat(CONTROLS.values(), ignore_index=True)
    CONTROL_MANIFEST.to_csv(DIAG / "control_manifest.csv", index=False)   # saved before any control engine
    display(CONTROL_MANIFEST.groupby(["condition", "class"]).size().unstack(fill_value=0))
repeats = 1 if DRY_RUN else int(dcfg["rebuild_repeats"])
JOBS = [(f"natural_s1_rebuild{r}", "natural", CAL) for r in range(repeats)]
JOBS += [(f"control_{name}_s1", name, manifest) for name, manifest in CONTROLS.items()]
preflight("pilot diagnostic engines", len(JOBS), [REF_ENGINE, CAL_PATH],
          [OUT.engines / "diagnostics", DIAG], LOCAL_WORK_ROOT,
          {"method": METHOD, "jobs": [j[0] for j in JOBS], "detailed_profile": True}, [1], DRY_RUN,
          "~4-7 min per engine (build + scoring)")

class,bona_fide,print,replay
condition,,,
bona_fide_only,600,0,0
tiny_natural,10,5,5


PRE-FLIGHT: pilot diagnostic engines   [FULL RUN]
Items to process : 5
Sources          : 
  /content/drive/MyDrive/QPAD_Project/engines/int8_entropy_natural_s1 
  /content/drive/MyDrive/QPAD_Project/results/int8/calibration_manifest.csv
Destinations     : 
  /content/drive/MyDrive/QPAD_Project/engines/diagnostics 
  /content/drive/MyDrive/QPAD_Project/results/int8/diagnostics
Local free space : 48.3 GB at /content/qpad_work
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   68G   45G  61% /
Estimate         : ~4-7 min per engine (build + scoring)
Seed             : [1]
Configuration    :
{
  "method": "entropy",
  "jobs": [
    "natural_s1_rebuild0",
    "natural_s1_rebuild1",
    "natural_s1_rebuild2",
    "control_bona_fide_only_s1",
    "control_tiny_natural_s1"
  ],
  "detailed_profile": true
}


In [21]:
RUNS = {}
for name, condition, manifest in JOBS:
    tag = f"diagnostics/int8_{METHOD}_{name}"
    edir, rdir = OUT.engines / tag, OUT.results / "int8" / tag
    if engine_is_complete(edir, rdir) and not FORCE_REBUILD:
        print("verified, skipping", tag)
    else:
        images = calibration_array(CAL_STORE, manifest, 1, condition, norm)
        build_int8_engine(paths, cfgs, DRY_RUN, images, tag, METHOD, detailed_profile=True)
        evaluate_backend(TrtRunner(edir / "engine.plan"), DEV, TEST, norm, FP32_THR, rdir, nfr,
                         int(icfg["tensorrt"]["opt_batch"]))
    RUNS[name] = (edir, rdir)
    m = json.loads((rdir / "metrics.json").read_text())["policy_B"]
    print(name, "| print", m["print_apcer"]["text"], "| replay", m["replay_apcer"]["text"], "| BPCER", m["bpcer"]["text"])

natural_s1_rebuild0 | print 3/360 = 0.83% | replay 31/360 = 8.61% | BPCER 7/360 = 1.94%
natural_s1_rebuild1 | print 5/360 = 1.39% | replay 33/360 = 9.17% | BPCER 7/360 = 1.94%
natural_s1_rebuild2 | print 3/360 = 0.83% | replay 30/360 = 8.33% | BPCER 7/360 = 1.94%
control_bona_fide_only_s1 | print 11/360 = 3.06% | replay 47/360 = 13.06% | BPCER 1/360 = 0.28%
control_tiny_natural_s1 | print 5/360 = 1.39% | replay 38/360 = 10.56% | BPCER 4/360 = 1.11%


## 1. How much of the network runs in INT8?
Per-layer report of the rebuilt natural seed-1 engine. A layer counts as INT8 when its kernel reads INT8 input. Format-conversion (Reformat) layers are listed but not counted. Fused graphs that TensorRT reports as a single `ForeignNode` (Myelin) hide their inner layers.

In [22]:
LAYERS = D.layer_precision_table(engine_layers(RUNS["natural_s1_rebuild0"][0] / "engine.plan"))
PRECISION = D.precision_summary(LAYERS)
LAYERS.to_csv(DIAG / "layer_precision_natural_s1.csv", index=False)
PRECISION.to_csv(DIAG / "precision_summary.csv", index=False)
display(PRECISION)
display(LAYERS.groupby(["type", "precision"]).size().unstack(fill_value=0))
print("fused ForeignNode/Myelin layers:", int(LAYERS["type"].str.contains("Foreign|Myelin", case=False).sum()))

,precision,layers,share_pct
0,INT8,63,92.647059
1,FP32,5,7.352941


precision,FP32,INT8
type,,
CaskConvolution,0,40
CaskGemmConvolution,1,0
CaskPooling,0,3
NoOp,2,0
PointWiseV2,2,0
Reformat,37,96
Scale,0,20


fused ForeignNode/Myelin layers: 0


## 2. Rebuild noise
Identical rebuilds of the natural seed-1 engine. Differences between them are chance (kernel selection), so a calibration-mix effect must be larger than this to count.

In [23]:
rebuilds = [RUNS[n] for n, _, _ in JOBS if n.startswith("natural_s1_rebuild")]
if len(rebuilds) > 1:
    RN = rebuild_noise([r for _, r in rebuilds], [e for e, _ in rebuilds],
                       float(icfg["rebuild_noise"]["max_metric_variation_pp"]))
    (DIAG / "rebuild_noise.json").write_text(json.dumps(RN, indent=2))
    print(json.dumps(RN, indent=1))
    SHIFT_RB, FLIPS_RB = D.compare_pairs([("pilot natural s1 vs rebuild0", REF_RESULT, rebuilds[0][1])])
    display(SHIFT_RB)
    display(FLIPS_RB)
else:
    print("dry run: one rebuild only, rebuild noise needs at least two")

{
 "n_builds": 3,
 "max_score_movement": 0.032921329178676295,
 "videos_with_decision_flip": 5,
 "metric_movement_pp": {
  "print_apcer": 0.5555555555555555,
  "replay_apcer": 0.8333333333333331,
  "apcer_pooled": 0.6944444444444448,
  "bpcer": 0.0,
  "acer": 0.3472222222222224
 },
 "exceeds_limit": true,
 "limit_pp": 0.5,
 "engine_sha256": [
  "5ddd86f7a8ba2365dd1d57a4a3a623e6d9b9b75f879c145cc0511138133eda47",
  "abbaedbd3b0391b42393cb992550eb1a6f7f54b3a7440b2d0afbfdd623a51c44",
  "0fe1bddeec19345409c68da305f9809c8dd7f0958587f1a9ec5e4ee0a5a6b069"
 ],
 "identical_engines": false,
 "thresholds": [
  0.7297600280797603,
  0.7405650503857404,
  0.7283725684318616
 ]
}


,comparison,class,mean_delta,mean_abs_delta,median_abs_delta,max_abs_delta,moved_over_0.01
0,pilot natural s1 vs rebuild0,bona_fide,-0.000287,0.000337,0.000008,0.007653,0
1,pilot natural s1 vs rebuild0,print,-0.001622,0.001645,0.000577,0.014610,3
2,pilot natural s1 vs rebuild0,replay,-0.001508,0.001644,0.001082,0.009391,0


,comparison,policy,attack_rejected_to_accepted,attack_accepted_to_rejected,bona_fide_accepted_to_rejected,bona_fide_rejected_to_accepted,n_videos
0,pilot natural s1 vs rebuild0,B,1,1,0,0,1080


## 3. Positive controls
If even these extreme calibration sets barely change the results, either the model is robust to calibration or little of it runs in INT8 (see section 1). If they do change results, the pipeline can detect a calibration effect and the pilot's null result is credible.

In [24]:
if CONTROLS:
    rows = [{**metrics_rows("pilot natural s1", json.loads((REF_RESULT / "metrics.json").read_text())["policy_B"])}]
    for name in CONTROLS:
        rdir = RUNS[f"control_{name}_s1"][1]
        rows.append(metrics_rows(f"control {name}", json.loads((rdir / "metrics.json").read_text())["policy_B"]))
    CONTROL_METRICS = pd.DataFrame(rows)
    CONTROL_METRICS.to_csv(DIAG / "control_metrics.csv", index=False)
    display(CONTROL_METRICS)
    SHIFT_C, FLIPS_C = D.compare_pairs([(f"control {n} vs natural s1", REF_RESULT, RUNS[f"control_{n}_s1"][1])
                                        for n in CONTROLS])
    SHIFT_C.to_csv(DIAG / "control_score_shifts.csv", index=False)
    FLIPS_C.to_csv(DIAG / "control_decision_flips.csv", index=False)
    display(SHIFT_C)
    display(FLIPS_C)
    scales = {}
    for tag, edir in [(REF_TAG, REF_ENGINE)] + [(f"control_{n}", RUNS[f"control_{n}_s1"][0]) for n in CONTROLS]:
        table = pd.read_csv(edir / "scales.csv")
        scales[tag] = dict(zip(table["tensor"], table["scale"]))
    CONTROL_SCALES = scale_summary(compare_scales(scales, REF_TAG))
    CONTROL_SCALES.to_csv(DIAG / "control_scale_audit.csv", index=False)
    display(CONTROL_SCALES)
else:
    print("dry run: positive controls are built in the full run only")

,model,threshold,print_apcer,replay_apcer,apcer_pooled,bpcer,acer_pct,acer_max_pai_pct,eer_pct,auroc
0,pilot natural s1,0.733314,3/360 = 0.83%,31/360 = 8.61%,34/720 = 4.72%,7/360 = 1.94%,3.333333,5.277778,3.055556,0.996026
1,control bona_fide_only,0.512718,11/360 = 3.06%,47/360 = 13.06%,58/720 = 8.06%,1/360 = 0.28%,4.166667,6.666667,2.500000,0.997473
2,control tiny_natural,0.593991,5/360 = 1.39%,38/360 = 10.56%,43/720 = 5.97%,4/360 = 1.11%,3.541667,5.833333,3.055556,0.996269


,comparison,class,mean_delta,mean_abs_delta,median_abs_delta,max_abs_delta,moved_over_0.01
0,control bona_fide_only vs natural s1,bona_fide,-0.001694,0.005250,0.000236,0.157358,53
1,control bona_fide_only vs natural s1,print,-0.017129,0.021489,0.004889,0.238085,148
2,control bona_fide_only vs natural s1,replay,-0.041108,0.043832,0.021778,0.243999,234
3,control tiny_natural vs natural s1,bona_fide,-0.005091,0.008630,0.000308,0.182357,58
4,control tiny_natural vs natural s1,print,-0.027239,0.030186,0.005051,0.502555,155
5,control tiny_natural vs natural s1,replay,-0.027497,0.037455,0.015373,0.290272,212


,comparison,policy,attack_rejected_to_accepted,attack_accepted_to_rejected,bona_fide_accepted_to_rejected,bona_fide_rejected_to_accepted,n_videos
0,control bona_fide_only vs natural s1,B,24,0,0,6,1080
1,control tiny_natural vs natural s1,B,12,3,0,3,1080


,engine,median_abs_rel_diff,p95_abs_rel_diff,max_abs_rel_diff,n_tensors
0,control_bona_fide_only,0.055586,0.291532,0.782079,132
1,control_tiny_natural,0.072107,0.260398,0.693608,132
2,int8_entropy_natural_s1,0.000000,0.000000,0.000000,132


**Next notebook:** `08_full_int8_experiment.ipynb`